# Решение

Для решения задачи я использовала модель градиентного бустинга CatBoost.

Сначала был построен бейзлайн на основе логистической регрессии, который показал значение метрики около `AP = 0.48`.

Затем я выполнила подбор гиперпараметров для логистической регрессии, однако это почти не повлияло на качество модели.

После этого я обучила CatBoost и подобрала гиперпараметры с помощью Grid Search. Модель показала более высокий результат `AP = 0.53`.

Наибольший прирост метрики удалось получить за счёт улучшения набора данных. Были добавлены новые признаки из файла `events.csv`, что повысило метрику до `AP = 0.64`. Затем признаки были доработаны и после этого итоговое значение метрики составило `AP = 0.73`.

Лучшей оказалась модель CatBoost со следующими параметрами:

```python
{
    "depth": 5,
    "l2_leaf_reg": 3,
    "learning_rate": 0.07
}



In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import GridSearchCV, PredefinedSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import clone

from catboost import CatBoostClassifier
from sklearn.model_selection import ParameterGrid


# Пути к данным.
NOTEBOOK_DIR = Path('..')
DATA_DIR = NOTEBOOK_DIR / "data"

TARGET = "target"

# Эти колонки не используем как признаки модели.
ID_COLUMNS = {"lead_id", "user_id"}
TIME_COLUMNS = {"assignment_ts", "assignment_date"}
NON_FEATURE_COLUMNS = ID_COLUMNS | TIME_COLUMNS | {TARGET, "split"}

RANDOM_STATE = 42

## Загрузка данных

Загружаем обучающую выборку, тестовую выборку и события.  
В baseline модель использует только `train.csv` и `test.csv`.

In [2]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
events = pd.read_csv(DATA_DIR / "events.csv")

print("train:", train.shape)
print("test:", test.shape)
print("events:", events.shape)
print("target mean:", train[TARGET].mean())

train: (13694, 119)
test: (4306, 118)
events: (254705, 7)
target mean: 0.20746312253541696


**Преобразуем время в datetime**

In [3]:
train = train.copy()
test = test.copy()

for dateframe in [train, test]:
    dateframe['assignment_date'] = pd.to_datetime(dateframe['assignment_date'], errors="coerce")
    dateframe['assignment_ts'] = pd.to_datetime(dateframe['assignment_ts'], errors="coerce")
    dateframe['assignment_ts_norm'] = dateframe['assignment_ts'].dt.normalize()
events['event_ts'] = pd.to_datetime(events['event_ts'], errors="coerce")

In [4]:
print(list(train.columns))
train.head()

['lead_id', 'user_id', 'assignment_ts', 'assignment_date', 'lead_source', 'call_center', 'region', 'car_segment', 'lead_channel', 'user_tenure_bucket', 'price_bucket', 'assignment_hour', 'assignment_weekday', 'is_weekend', 'user_active_days_30d', 'user_age_days', 'prior_assignments_30d', 'seller_inventory_count', 'seller_response_rate_30d', 'item_price_log', 'car_age_years', 'mileage_km_log', 'item_views_1d', 'item_views_3d', 'item_views_7d', 'item_views_14d', 'item_views_30d', 'item_views_90d', 'item_favorites_1d', 'item_favorites_3d', 'item_favorites_7d', 'item_favorites_14d', 'item_favorites_30d', 'item_favorites_90d', 'detail_expands_1d', 'detail_expands_3d', 'detail_expands_7d', 'detail_expands_14d', 'detail_expands_30d', 'detail_expands_90d', 'photo_swipes_1d', 'photo_swipes_3d', 'photo_swipes_7d', 'photo_swipes_14d', 'photo_swipes_30d', 'photo_swipes_90d', 'seller_page_views_1d', 'seller_page_views_3d', 'seller_page_views_7d', 'seller_page_views_14d', 'seller_page_views_30d', 's

,lead_id,user_id,assignment_ts,assignment_date,lead_source,call_center,region,car_segment,lead_channel,user_tenure_bucket,...,leadgen_prev_positive_30d,leadgen_prev_positive_90d,active_days_auto_1d,active_days_auto_3d,active_days_auto_7d,active_days_auto_14d,active_days_auto_30d,active_days_auto_90d,target,assignment_ts_norm
0,lead_f57db09ab39ae3e7,user_0000001,2026-04-22 11:56:00,2026-04-22,CRM,external,west,budget,retargeting,warm,...,0.0,0.0,0.0,0.0,0.0,4.0,9.0,26.0,0,2026-04-22
1,lead_a6184b8a8165a27b,user_0000002,2026-04-07 14:49:00,2026-04-07,CRM,voxys,north,standard,partner,warm,...,NaN,1.0,0.0,0.0,0.0,NaN,4.0,5.0,0,2026-04-07
2,lead_229c2a117dbac203,user_0000003,2026-04-12 17:01:00,2026-04-12,Perf,external,north,budget,retargeting,new,...,0.0,NaN,2.0,4.0,1.0,10.0,12.0,52.0,0,2026-04-12
3,lead_16b19e58042ef905,user_0000005,2026-04-13 21:39:00,2026-04-13,Model,voxys,east,premium,partner,warm,...,1.0,0.0,0.0,1.0,0.0,3.0,2.0,NaN,1,2026-04-13
4,lead_734c227324978a36,user_0000006,2026-04-12 18:01:00,2026-04-12,CRM,voxys,central,budget,retargeting,warm,...,0.0,0.0,0.0,0.0,0.0,1.0,2.0,9.0,0,2026-04-12


In [5]:
events.head()

,lead_id,user_id,event_ts,event_type,item_price_log,src_slot,ctx_seq
0,lead_00025e9610a0d90d,user_0016636,2026-03-27 06:41:00,chat_open,13.303438,19.0,c02
1,lead_00025e9610a0d90d,user_0016636,2026-03-31 09:10:00,item_view,13.322707,6.0,c06
2,lead_00025e9610a0d90d,user_0016636,2026-04-02 22:04:00,item_view,13.395721,10.0,c06
3,lead_00025e9610a0d90d,user_0016636,2026-04-04 09:19:00,search,13.395955,10.0,c04
4,lead_00025e9610a0d90d,user_0016636,2026-04-07 12:36:00,item_view,13.472769,2.0,c02


## Анализ даных

In [6]:
print("Доля положительного класса:", round(train[TARGET].mean(), 4))

Доля положительного класса: 0.2075


In [7]:
dtype_report = train.dtypes.astype(str).value_counts().rename("columns_count")
display(dtype_report.to_frame())

,columns_count
float64,107
str,9
datetime64[us],3
int64,1


In [8]:
print("Самые частые события:")
display(events["event_type"].value_counts(dropna=False).to_frame("count"))

print("Время:")
print(events["event_ts"].min(), "—", events["event_ts"].max())

print("Доля кол-ва пропусков:")
display(events.isna().mean().sort_values(ascending=False).to_frame("missing_share"))

Самые частые события:


,count
event_type,
item_view,120905
search,61101
favorite,26333
chat_open,24797
call_click,21569


Время:
2026-03-08 10:03:00 — 2026-04-27 20:28:00
Доля кол-ва пропусков:


,missing_share
lead_id,0.0
user_id,0.0
event_ts,0.0
event_type,0.0
item_price_log,0.0
src_slot,0.0
ctx_seq,0.0


## Feature engineering

### Добавление новых признаков из даты 
Тк большинство признаков которые можно извлечь из даты уже есть в датафрейме, я добавила временной тренд и циклическое кодирование часа и дня недели с помощью `sin` и `cos`

In [9]:
reference_date = (pd.to_datetime(train['assignment_date']).min().normalize())

print(f'Исходная дата : {reference_date}\n')


def add_data_features(dataframe, reference_date=reference_date):
    result = dataframe.copy()

    assignment_date = (pd.to_datetime(result['assignment_date']).dt.normalize())

    hour = result["assignment_hour"]
    weekday = result["assignment_weekday"]

    result["days_from_start"] = (assignment_date - reference_date).dt.days.astype("int16")

    result["assignment_hour_sin"] = np.sin(2 * np.pi * hour / 24)
    result["assignment_hour_cos"] = np.cos(2 * np.pi * hour / 24)

    result["assignment_weekday_sin"] = np.sin(2 * np.pi * weekday / 7)
    result["assignment_weekday_cos"] = np.cos(2 * np.pi * weekday / 7)
    
    return result


train = add_data_features(train)

test = add_data_features(test)

Исходная дата : 2026-04-07 00:00:00



### Добавление новых признаков из файла `events.csv`

Из файла `events.csv` были добавлены новые признаки, отражающие частоту и давность действий пользователя, его активность в нескольких временных окнах, характеристики последних событий, уровень интереса и сходство цен просмотренных объектов с ценой текущего объявления


In [10]:
def add_event_features(train, test, events, max_history_days=30):
    train_df = train.copy()
    test_df = test.copy()
    events_df = events.copy()
    
    train_assignments = train_df[["lead_id", "assignment_ts", "item_price_log"]].copy()
    test_assignments = test_df[["lead_id", "assignment_ts", "item_price_log"]].copy()
    
    assignments = pd.concat([train_assignments, test_assignments], ignore_index=True)
    assignments = assignments.drop_duplicates(subset=["lead_id"], keep="first")
    assignments = assignments.rename(
        columns={
            "item_price_log": "lead_item_price_log",
        }
    )

    ev = events_df.merge(assignments, on="lead_id", how="inner")
    ev["age_hours"] = (ev["assignment_ts"] - ev["event_ts"]).dt.total_seconds() / 3600

    ev = ev[
        (ev["age_hours"] > 0)
        & (ev["age_hours"] <= max_history_days * 24)
    ].copy()

    ev["event_day"] = ev["event_ts"].dt.floor("D")
    ev["event_hour"] = ev["event_ts"].dt.hour
    ev["price_diff"] = ev["item_price_log"] - ev["lead_item_price_log"]
    ev["abs_price_diff"] = ev["price_diff"].abs()
    ev = ev.sort_values(["lead_id", "event_ts"])

    event_types = sorted(ev["event_type"].dropna().unique())
    parts = []

    windows = [
        (1, "1h"), (3, "3h"), (6, "6h"), (12, "12h"),
        (24, "1d"), (72, "3d"), (168, "7d"),
        (336, "14d"), (720, "30d"),
    ]

    for hours, suffix in windows:
        recent = ev[ev["age_hours"] <= hours]

        parts.append(
            recent.groupby("lead_id").size().rename(f"ev_count_{suffix}")
        )

        counts = (
            recent.groupby(["lead_id", "event_type"])
            .size()
            .unstack(fill_value=0)
            .reindex(columns=event_types, fill_value=0)
        )
        counts.columns = [
            f"ev_{event_type}_count_{suffix}"
            for event_type in counts.columns
        ]
        parts.append(counts)

        if hours in (24, 72, 168, 720):
            parts.extend([
                recent.groupby("lead_id")["event_day"]
                    .nunique().rename(f"ev_active_days_{suffix}"),
                recent.groupby("lead_id")["event_type"]
                    .nunique().rename(f"ev_unique_types_{suffix}"),
                recent.groupby("lead_id")["ctx_seq"]
                    .nunique().rename(f"ev_unique_ctx_{suffix}"),
                recent.groupby("lead_id")["src_slot"]
                    .nunique().rename(f"ev_unique_slots_{suffix}"),
            ])

    reversed_events = ev.sort_values(
        ["lead_id", "event_ts"], ascending=[True, False]
    ).copy()
    reversed_events["rank"] = reversed_events.groupby("lead_id").cumcount() + 1

    for rank in (1, 2, 3):
        last = reversed_events[reversed_events["rank"] == rank].set_index("lead_id")
        prefix = "ev_last" if rank == 1 else f"ev_last{rank}"

        parts.extend([
            last["age_hours"].rename(f"{prefix}_age_hours"),
            last["event_type"].rename(f"{prefix}_type"),
            last["ctx_seq"].rename(f"{prefix}_ctx"),
            last["src_slot"].rename(f"{prefix}_slot"),
            last["price_diff"].rename(f"{prefix}_price_diff"),
            last["abs_price_diff"].rename(f"{prefix}_abs_price_diff"),
            last["event_hour"].rename(f"{prefix}_hour"),
        ])

    for event_type in event_types:
        typed = ev[ev["event_type"] == event_type]
        last = typed.groupby("lead_id").tail(1).set_index("lead_id")
        parts.extend([
            last["age_hours"].rename(f"ev_{event_type}_recency_hours"),
            last["price_diff"].rename(f"ev_{event_type}_last_price_diff"),
        ])

    intent_weights = {
        "search": 0.25,
        "item_view": 1.0,
        "favorite": 3.0,
        "chat_open": 2.5,
        "call_click": 2.5,
    }
    ev["intent_weight"] = ev["event_type"].map(intent_weights).fillna(0.0)

    for tau in (6, 24, 72, 168):
        decay = np.exp(-ev["age_hours"] / tau)

        parts.append(
            decay.groupby(ev["lead_id"])
            .sum()
            .rename(f"ev_decay_count_{tau}h")
        )
        parts.append(
            (decay * ev["intent_weight"])
            .groupby(ev["lead_id"])
            .sum()
            .rename(f"ev_intent_decay_{tau}h")
        )

        for event_type in event_types:
            mask = ev["event_type"] == event_type
            parts.append(
                decay[mask]
                .groupby(ev.loc[mask, "lead_id"])
                .sum()
                .rename(f"ev_{event_type}_decay_{tau}h")
            )

    for threshold, name in ((0.03, "003"), (0.10, "010"), (0.25, "025")):
        matched = ev[ev["abs_price_diff"] <= threshold]
        last = matched.groupby("lead_id").tail(1).set_index("lead_id")

        parts.append(
            last["age_hours"].rename(f"ev_price_match_{name}_recency_hours")
        )

        for hours, suffix in ((24, "1d"), (72, "3d"), (168, "7d"), (720, "30d")):
            recent = matched[matched["age_hours"] <= hours]
            parts.append(
                recent.groupby("lead_id")
                .size()
                .rename(f"ev_price_match_{name}_count_{suffix}")
            )

            high_intent = recent[
                recent["event_type"].isin(["favorite", "chat_open", "call_click"])
            ]
            parts.append(
                high_intent.groupby("lead_id")
                .size()
                .rename(f"ev_price_match_{name}_high_intent_{suffix}")
            )

    for hours, suffix in ((72, "3d"), (168, "7d"), (720, "30d")):
        recent = ev[ev["age_hours"] <= hours]
        parts.append(
            recent.groupby("lead_id").agg(
                **{
                    f"ev_price_diff_mean_{suffix}": ("price_diff", "mean"),
                    f"ev_price_diff_std_{suffix}": ("price_diff", "std"),
                    f"ev_price_diff_absmean_{suffix}": ("abs_price_diff", "mean"),
                    f"ev_price_diff_minabs_{suffix}": ("abs_price_diff", "min"),
                    f"ev_slot_mean_{suffix}": ("src_slot", "mean"),
                    f"ev_slot_std_{suffix}": ("src_slot", "std"),
                    f"ev_slot_min_{suffix}": ("src_slot", "min"),
                    f"ev_slot_max_{suffix}": ("src_slot", "max"),
                }
            )
        )

    ctx_counts = (
        ev.groupby(["lead_id", "ctx_seq"])
        .size()
        .unstack(fill_value=0)
    )
    ctx_counts.columns = [f"ev_ctx_{ctx}_count_30d" for ctx in ctx_counts.columns]
    parts.append(ctx_counts)

    event_features = pd.concat(parts, axis=1)
    event_features = event_features.loc[:, ~event_features.columns.duplicated()]
    event_features = event_features.reindex(assignments["lead_id"])

    zero_columns = [
        column for column in event_features.columns
        if (
            "_count_" in column
            or column.startswith("ev_count_")
            or "active_days" in column
            or "unique_" in column
            or "decay_" in column
            or "high_intent_" in column
        )
    ]
    event_features[zero_columns] = event_features[zero_columns].fillna(0)

    extra = pd.DataFrame(index=event_features.index)
    extra["ev_gap_last1_last2_hours"] = (
        event_features["ev_last2_age_hours"]
        - event_features["ev_last_age_hours"]
    )
    extra["ev_gap_last2_last3_hours"] = (
        event_features["ev_last3_age_hours"]
        - event_features["ev_last2_age_hours"]
    )
    extra["ev_last_hour_sin"] = np.sin(
        2 * np.pi * event_features["ev_last_hour"] / 24
    )
    extra["ev_last_hour_cos"] = np.cos(
        2 * np.pi * event_features["ev_last_hour"] / 24
    )
    event_features = pd.concat([event_features, extra], axis=1)

    event_features = event_features.reset_index().rename(columns={"index": "lead_id"})

    train_result = train_df.merge(event_features, on="lead_id", how="left")
    test_result = test_df.merge(event_features, on="lead_id", how="left")

    for dataframe in (train_result, test_result):
        dataframe["ev_recent_share_1d_30d"] = (
            dataframe["ev_count_1d"] / (dataframe["ev_count_30d"] + 1)
        )
        dataframe["ev_recent_share_3d_30d"] = (
            dataframe["ev_count_3d"] / (dataframe["ev_count_30d"] + 1)
        )
        dataframe["ev_recent_share_7d_30d"] = (
            dataframe["ev_count_7d"] / (dataframe["ev_count_30d"] + 1)
        )

        for event_type in event_types:
            dataframe[f"ev_{event_type}_share_7d"] = (
                dataframe[f"ev_{event_type}_count_7d"]
                / (dataframe["ev_count_7d"] + 1)
            )
            dataframe[f"ev_{event_type}_share_30d"] = (
                dataframe[f"ev_{event_type}_count_30d"]
                / (dataframe["ev_count_30d"] + 1)
            )

    return train_result, test_result, event_features


old_columns = set(train.columns)
train, test, new_features =  add_event_features(train, test, events)

new_event_columns = sorted(set(train.columns) - old_columns)
print("Новых признаков:", len(new_event_columns))

Новых признаков: 205


/var/folders/jx/ld7mb5ld4472ztzr9xwhz2_40000gn/T/ipykernel_61419/509341841.py:215: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  event_features = event_features.reset_index().rename(columns={"index": "lead_id"})
/var/folders/jx/ld7mb5ld4472ztzr9xwhz2_40000gn/T/ipykernel_61419/509341841.py:221: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataframe["ev_recent_share_1d_30d"] = (
/var/folders/jx/ld7mb5ld4472ztzr9xwhz2_40000gn/T/ipykernel_61419/509341841.py:224: PerformanceWarning: DataFrame is highly fragmented.  This is usually t

## Предобработка данных

In [11]:
feature_columns = [column
    for column in train.columns
    if column not in NON_FEATURE_COLUMNS
    and column in test.columns
]

numeric_columns = [
    column
    for column in feature_columns
    if pd.api.types.is_numeric_dtype(train[column])
]

categorical_columns = [
    column
    for column in feature_columns
    if column not in numeric_columns
]

print("Числовых признаков:", len(numeric_columns))
print("Категориальных признаков:", len(categorical_columns))
print("Всего признаков до one-hot:", len(feature_columns))
print("Категориальные признаки:", sorted(categorical_columns))

Числовых признаков: 311
Категориальных признаков: 14
Всего признаков до one-hot: 325
Категориальные признаки: ['assignment_ts_norm', 'call_center', 'car_segment', 'ev_last2_ctx', 'ev_last2_type', 'ev_last3_ctx', 'ev_last3_type', 'ev_last_ctx', 'ev_last_type', 'lead_channel', 'lead_source', 'price_bucket', 'region', 'user_tenure_bucket']


In [12]:
numeric_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median", add_indicator=True),
        ),
        ("scaler", StandardScaler()),
    ]
)

categorical_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                dtype=np.float32,
            ),
        ),
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocessor, numeric_columns),
        ("cat", categorical_preprocessor, categorical_columns),
    ],
    remainder="drop",
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

## Валидация

Так как тестовая выборка находится позже train по времени, лучше валидироваться на последних датах train.  
Это ближе к реальному сценарию, чем случайный split.

In [13]:
def make_validation_split(train_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Делит train по времени: ранние даты в обучение, поздние даты в валидацию."""
    if "assignment_date" in train_df.columns:
        dates = pd.to_datetime(train_df["assignment_date"]).dt.date
        ordered_dates = sorted(dates.unique())
        cutoff = ordered_dates[int(len(ordered_dates) * 0.8)]

        train_part = train_df[dates < cutoff]
        valid_part = train_df[dates >= cutoff]
        return train_part, valid_part

    return train_test_split(
        train_df,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=train_df[TARGET],
    )


train_part, valid_part = make_validation_split(train)

print("train_part:", train_part.shape)
print("valid_part:", valid_part.shape)

train_part: (10272, 330)
valid_part: (3422, 330)


## Модель Logistic Regression

### Подбор гиперпараметров

In [14]:
# logreg_pipeline = Pipeline(
#     steps=[
#         ("preprocessor", clone(preprocessor)),
#         (
#             "model",
#             LogisticRegression(
#                 solver="liblinear",
#                 max_iter=3000,
#                 random_state=RANDOM_STATE,
#             ),
#         ),
#     ]
# )

# search_data = pd.concat(
#     [train_part, valid_part],
#     ignore_index=True,
# )

# X_search = search_data[feature_columns]
# y_search = search_data[TARGET]

# test_fold = np.concatenate(
#     [
#         np.full(len(train_part), -1, dtype=int),
#         np.zeros(len(valid_part), dtype=int),
#     ]
# )

# predefined_split = PredefinedSplit(test_fold=test_fold)

In [15]:
# param_grid = {
#     "model__C": [
#         0.001,
#         0.003,
#         0.01,
#         0.03,
#         0.1,
#         0.3,
#         1.0,
#         3.0,
#         10.0,
#         30.0,
#         100.0,
#     ],

#     "model__l1_ratio": [
#         0.0,
#         1.0,
#     ],

#     "model__class_weight": [
#         None,
#         "balanced",
#     ],
# }

# grid_search = GridSearchCV(
#     estimator=logreg_pipeline,
#     param_grid=param_grid,
#     scoring="average_precision",
#     cv=predefined_split,
#     n_jobs=-1,
#     verbose=1,
#     refit=False,
# )

# grid_search.fit(X_search, y_search)

# best_params = grid_search.best_params_

# print("Лучшие параметры:", best_params)
# print(f"Лучший AP: {grid_search.best_score_:.5f}")


### Финальная модель

In [16]:
# best_model = clone(logreg_pipeline).set_params(**best_params)

# best_model

In [17]:
# grid_results = (
#     pd.DataFrame(grid_search.cv_results_)
#     .sort_values("rank_test_score")
# )

# display(
#     grid_results[
#         [
#             "rank_test_score",
#             "mean_test_score",
#             "std_test_score",
#             "param_model__C",
#             "param_model__l1_ratio",
#             "param_model__class_weight",
#         ]
#     ].head(10)
# )

In [18]:
# best_model.fit(
#     train_part[feature_columns],
#     train_part[TARGET],
# )

# valid_scores = best_model.predict_proba(
#     valid_part[feature_columns]
# )[:, 1]

# valid_ap = average_precision_score(
#     valid_part[TARGET],
#     valid_scores,
# )

# print(f"Average Precision: {valid_ap:.5f}")

## Модель CatBoost

### Подбор гиперпараметров

In [19]:
def daily_average_precision(dataframe, predictions):
    metric_data = pd.DataFrame({
        "assignment_date": pd.to_datetime(
            dataframe["assignment_date"]
        ).dt.date,
        "target": dataframe[TARGET].to_numpy(),
        "score": predictions,
    })

    daily_scores = []

    for _, day_data in metric_data.groupby("assignment_date"):
        if day_data["target"].sum() == 0:
            daily_scores.append(0.0)
        else:
            daily_scores.append(
                average_precision_score(
                    day_data["target"],
                    day_data["score"],
                )
            )

    return float(np.mean(daily_scores))


datetime_columns = [
    column
    for column in feature_columns
    if pd.api.types.is_datetime64_any_dtype(train[column])
]

catboost_feature_columns = [
    column
    for column in feature_columns
    if column not in datetime_columns
]

catboost_categorical_columns = [
    column
    for column in categorical_columns
    if column in catboost_feature_columns
]


def prepare_catboost_data(dataframe):
    result = dataframe[catboost_feature_columns].copy()

    for column in catboost_categorical_columns:
        result[column] = (
            result[column]
            .astype("object")
            .where(result[column].notna(), "__missing__")
            .astype(str)
        )

    return result


X_train_cb = prepare_catboost_data(train_part)
X_valid_cb = prepare_catboost_data(valid_part)

y_train_cb = train_part[TARGET]
y_valid_cb = valid_part[TARGET]

In [ ]:
param_grid = {
    "depth": [5, 7],
    "learning_rate": [0.03, 0.07],
    "l2_leaf_reg": [3, 8],
}

search_results = []

best_catboost_score = -np.inf
best_catboost_params = None
best_catboost_iterations = None


for params in ParameterGrid(param_grid):
    current_model = CatBoostClassifier(
        iterations=1000,
        loss_function="Logloss",
        eval_metric="PRAUC:type=Classic",
        random_seed=RANDOM_STATE,
        verbose=False,
        allow_writing_files=False,
        **params,
    )

    current_model.fit(
        X_train_cb,
        y_train_cb,
        cat_features=catboost_categorical_columns,
        eval_set=(X_valid_cb, y_valid_cb),
        early_stopping_rounds=80,
        use_best_model=True,
    )

    valid_predictions = current_model.predict_proba(
        X_valid_cb
    )[:, 1]

    current_score = daily_average_precision(
        valid_part,
        valid_predictions,
    )

    current_iterations = current_model.tree_count_

    search_results.append(
        {
            **params,
            "iterations": current_iterations,
            "daily_ap": current_score,
        }
    )

    print(
        params,
        f"iterations={current_iterations}",
        f"AP={current_score:.5f}",
    )

    if current_score > best_catboost_score:
        best_catboost_score = current_score
        best_catboost_params = params
        best_catboost_iterations = current_iterations

{'depth': 5, 'l2_leaf_reg': 3, 'learning_rate': 0.03} iterations=1000 AP=0.73516
{'depth': 5, 'l2_leaf_reg': 3, 'learning_rate': 0.07} iterations=774 AP=0.73943


In [ ]:
catboost_results = (
    pd.DataFrame(search_results)
    .sort_values("daily_ap", ascending=False)
    .reset_index(drop=True)
)

display(catboost_results)

print("Лучшие параметры:", best_catboost_params)
print("Кол-во деревьев:", best_catboost_iterations)

### Финальная модель

In [ ]:
X_full_cb = prepare_catboost_data(train)
X_test_cb = prepare_catboost_data(test)

model = CatBoostClassifier(
    iterations=best_catboost_iterations,
    loss_function="Logloss",
    random_seed=RANDOM_STATE,
    verbose=False,
    allow_writing_files=False,
    **best_catboost_params,
)

model.fit(
    X_full_cb,
    train[TARGET],
    cat_features=catboost_categorical_columns,
)

## Submission

Обучаем модель на всем train и строим score для test.  
Файл для отправки должен содержать две колонки: `lead_id` и `score`.

In [ ]:
test_scores = model.predict_proba(X_test_cb)[:, 1]

submission = pd.DataFrame(
    {
        "lead_id": test["lead_id"].astype(str),
        "score": test_scores,
    }
)

submission.to_csv(DATA_DIR / "submission.csv", index=False)
submission.head()

In [ ]:
# Минимальные проверки формата перед загрузкой.
assert list(submission.columns) == ["lead_id", "score"]
assert len(submission) == len(test)
assert submission["lead_id"].is_unique
assert submission["score"].between(0, 1).all()

print("submission.csv is ready")